# NB7 — Correlation Plots

In [ ]:
import os

if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

import pandas as pd
import numpy as np
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from itertools import combinations

TABLES     = "outputs/tables/"
GRAPH_ROOT = "outputs/graphs/"
MASTER     = "Master_Eval_Sheet.xlsx"

def _set_font():
    tnr = [f.name for f in fm.fontManager.ttflist if "Times New Roman" in f.name]
    plt.rcParams["font.serif"] = (["Times New Roman"] if tnr else []) + ["Liberation Serif","DejaVu Serif","FreeSerif"]
    plt.rcParams["font.family"] = "serif"
_set_font()

plt.rcParams.update({"font.size":10,"axes.titlesize":11,"axes.labelsize":10,
    "xtick.labelsize":9,"ytick.labelsize":9,"legend.fontsize":9,"figure.dpi":150})

TASKS=["C1","C2","C3","C4"]
TASK_LABELS={"C1":"Detection (C1)","C2":"Span+Description (C2)","C3":"Category (C3)","C4":"Correction (C4)"}
GROUP_ORDER=["G1_Humans","G2_Annotator_LLMs","G3_NonAnnotator_LLMs","G4_All_LLMs","G5_Humans+Ann_LLMs","G6_Humans+NonAnn_LLMs","G7_All"]
GROUP_LABELS={"G1_Humans":"Humans\n(H1+H2)","G2_Annotator_LLMs":"Annotator\nLLMs","G3_NonAnnotator_LLMs":"Non-Annotator\nLLMs","G4_All_LLMs":"All LLMs","G5_Humans+Ann_LLMs":"Humans +\nAnnot. LLMs","G6_Humans+NonAnn_LLMs":"Humans +\nNon-Annot. LLMs","G7_All":"All Raters"}
RATER_ORDER=["H1","H2","L1","L2","L3","L4"]
RATER_LABELS={"H1":"Human 1","H2":"Human 2","L1":"Sarvam-105B","L2":"LLaMA-4-Scout-17B","L3":"GPT-Oss-120B","L4":"Gemini-3.1-Flash-Lite-Preview"}
RATER_SHORT={"H1":"H1","H2":"H2","L1":"Sarvam","L2":"Llama","L3":"GPT","L4":"Gemini"}
CATEGORIES=["Script Normalization","Spelling & Typographical Error","Grammatical Error","Code-Mixing / Wrong Language","Correct Sentence / No Errors"]
CAT_SHORT={"Script Normalization":"Script\nNorm.","Spelling & Typographical Error":"Spelling","Grammatical Error":"Grammatical","Code-Mixing / Wrong Language":"Code-Mixing","Correct Sentence / No Errors":"Correct"}
EVAL_ORDER=["Human 1","Human 2","Sarvam-105B","LLaMA-4-Scout-17B","GPT-Oss-120B","Gemini-3.1-Flash-Lite-Preview"]
TASK_NB8=["Detection (C1)","Span+Description (C2)","Category (C3)","Correction (C4)"]
CATS_NB8=["Script\nNorm.","Spelling","Grammatical","Code-Mixing","Correct"]
llm_r=["L1","L2","L3","L4"]
llm_colors={"L1":"#4472C4","L2":"#ED7D31","L3":"#70AD47","L4":"#9E480E"}
llm_names=[RATER_LABELS[r] for r in llm_r]
COLORS4=["#4472C4","#ED7D31","#70AD47","#9E480E"]
COLORS2=["#4472C4","#ED7D31"]
COLORS6=["#4472C4","#ED7D31","#70AD47","#9E480E","#7030A0","#FF0000"]
COLORS7=["#4472C4","#ED7D31","#70AD47","#9E480E","#7030A0","#FF0000","#00B0F0"]
groups=     [GROUP_LABELS[g] for g in GROUP_ORDER]
rater_names=[RATER_LABELS[r] for r in RATER_ORDER]
cats_short= [CAT_SHORT[c] for c in CATEGORIES]
col_names=["#","Sentence_ID","Source_File","Source_Sentence","Gold_Category","LLM","Has_Errors","Error_Span","Annotated_Category","Description","Corrected_Sentence","H1_C1","H1_C2","H1_C3","H1_C4","H1_Total","H2_C1","H2_C2","H2_C3","H2_C4","H2_Total","L1_C1","L1_C2","L1_C3","L1_C4","L1_Total","L2_C1","L2_C2","L2_C3","L2_C4","L2_Total","L3_C1","L3_C2","L3_C3","L3_C4","L3_Total","L4_C1","L4_C2","L4_C3","L4_C4","L4_Total"]

def ab(ax,bars,vals,fmt=".1f",fs=7):
    ylim=ax.get_ylim(); yr=ylim[1]-ylim[0]; off=yr*0.018
    for bar,val in zip(bars,vals):
        try: v=float(val)
        except: continue
        if math.isnan(v): continue
        y=bar.get_height()
        yo=np.clip(y+off if v>=0 else y-off*2,ylim[0]+yr*0.01,ylim[1]-yr*0.05)
        ax.text(bar.get_x()+bar.get_width()/2,yo,f"{v:{fmt}}",ha="center",va="bottom" if v>=0 else "top",fontsize=fs,fontweight="normal")

def save_fig(fig,folder,filename):
    os.makedirs(folder,exist_ok=True)
    fig.savefig(os.path.join(folder,filename+".svg"),format="svg",bbox_inches="tight")
    fig.savefig(os.path.join(folder,filename+".pdf"),format="pdf",bbox_inches="tight")
    plt.close(fig); print("Saved:",filename)

def hmap(td,rows_l,cols_l,col,ax,title,vmin=-1,vmax=1):
    mat=pd.DataFrame(np.nan,index=rows_l,columns=cols_l)
    for _,row in td.iterrows():
        r1=RATER_LABELS.get(str(row.get("Rater1","")),str(row.get("Rater1","")))
        r2=RATER_LABELS.get(str(row.get("Rater2","")),str(row.get("Rater2","")))
        if r1 in rows_l and r2 in cols_l: mat.loc[r1,r2]=row[col]
        if r2 in rows_l and r1 in cols_l: mat.loc[r2,r1]=row[col]
    if rows_l==cols_l:
        m2=mat.values.copy().astype(float); np.fill_diagonal(m2,1.0); mat=pd.DataFrame(m2,index=rows_l,columns=cols_l)
    sns.heatmap(mat.astype(float),annot=True,fmt=".2f",cmap="coolwarm",vmin=vmin,vmax=vmax,linewidths=0.5,ax=ax,annot_kws={"size":8},square=True)
    ax.set_title(title,fontsize=10); ax.set_xticklabels(ax.get_xticklabels(),rotation=45,ha="right",fontsize=8); ax.set_yticklabels(ax.get_yticklabels(),rotation=0,fontsize=8)

def cat_hmap(df,group,col,ax,title,task_filter=None,vmin=-1,vmax=1):
    sub=df[df["Group"]==group].copy()
    if task_filter: sub=sub[sub["Task"].isin(task_filter)]
    sub["Cat"]=sub["Category"].map(CAT_SHORT); sub["TL"]=sub["Task"].map(TASK_LABELS)
    tl_order=[TASK_LABELS[t] for t in (task_filter if task_filter else TASKS) if TASK_LABELS[t] in sub["TL"].values]
    pivot=sub.pivot_table(index="Cat",columns="TL",values=col,aggfunc="mean")
    pivot=pivot.reindex(index=cats_short,columns=tl_order)
    sns.heatmap(pivot,annot=True,fmt=".2f",cmap="coolwarm",vmin=vmin,vmax=vmax,linewidths=0.5,ax=ax,annot_kws={"size":9},square=True)
    ax.set_title(title,fontsize=10); ax.set_xlabel("Task"); ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(),rotation=30,ha="right",fontsize=9); ax.set_yticklabels(ax.get_yticklabels(),rotation=0,fontsize=9)

def grp_bar(df,val_col,ylabel,title,ylim,fmt=".2f"):
    fig,ax=plt.subplots(figsize=(13,6)); w=0.2; x=list(range(len(groups)))
    for ti,task in enumerate(TASKS):
        sub=df[df["Task"]==task].copy(); sub["GL"]=sub["Group"].map(GROUP_LABELS); vals=sub.set_index("GL").reindex(groups)[val_col].tolist()
        bars=ax.bar([xi+ti*w for xi in x],vals,w,label=TASK_LABELS[task],color=COLORS4[ti]); ab(ax,bars,vals,fmt=fmt,fs=7)
    ax.set_xticks([xi+1.5*w for xi in x]); ax.set_xticklabels(groups,fontsize=9); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(title="Task",loc="upper right"); ax.axhline(0,color="black",lw=0.8,ls="--"); ax.set_ylim(*ylim); plt.tight_layout(); return fig,ax

def dual_bar(df,c1,c2,l1,l2,ylabel,title,ylim,tasks=["C2","C4"],fmt=".2f"):
    fig,axes=plt.subplots(1,2,figsize=(14,6))
    for ai,task in enumerate(tasks):
        ax=axes[ai]; w=0.38; x=list(range(len(groups)))
        sub=df[df["Task"]==task].copy(); sub["GL"]=sub["Group"].map(GROUP_LABELS); sub=sub.set_index("GL").reindex(groups)
        v1=sub[c1].tolist(); v2=sub[c2].tolist()
        b1=ax.bar([xi-w/2 for xi in x],v1,w,label=l1,color=COLORS2[0]); b2=ax.bar([xi+w/2 for xi in x],v2,w,label=l2,color=COLORS2[1])
        ab(ax,b1,v1,fmt=fmt,fs=7); ab(ax,b2,v2,fmt=fmt,fs=7)
        ax.set_xticks(x); ax.set_xticklabels(groups,fontsize=8); ax.set_ylabel(ylabel); ax.set_title(f"{TASK_LABELS[task]}\n{title}")
        ax.legend(loc="upper right",fontsize=9); ax.axhline(0,color="black",lw=0.8,ls="--"); ax.set_ylim(*ylim)
    plt.tight_layout(pad=2); return fig

print("Setup complete.")


Mounted at /content/drive
Setup complete.


In [ ]:
FOLDER=GRAPH_ROOT+"NB7_Correlation/"
overall_df=pd.read_excel(TABLES+"NB7_Correlation_Results.xlsx",sheet_name="Overall").dropna(how="all")
category_df=pd.read_excel(TABLES+"NB7_Correlation_Results.xlsx",sheet_name="By_Category").dropna(how="all")
nb7o=overall_df; nb7c=category_df
mdf=pd.read_excel(MASTER,sheet_name="Master Eval Sheet",header=None,skiprows=2,names=col_names)
mdf=mdf[mdf["#"].notna()].copy(); mdf=mdf[mdf["#"].astype(str)!="TOTAL"].reset_index(drop=True)
print("Loaded NB7:", overall_df.shape)

Loaded NB7: (60, 8)


In [ ]:
nb7o=pd.read_excel(TABLES+"NB7_Correlation_Results.xlsx",sheet_name="Overall").dropna(how="all")
nb7c=pd.read_excel(TABLES+"NB7_Correlation_Results.xlsx",sheet_name="By_Category").dropna(how="all")
fig,axes=plt.subplots(1,4,figsize=(26,7))
for ai,task in enumerate(TASKS): hmap(nb7o[nb7o["Task"]==task],rater_names,rater_names,"Spearman_rho",axes[ai],f"{TASK_LABELS[task]}\nSpearman \u03c1")
plt.suptitle("Spearman Rank Correlation (\u03c1) — All Raters",y=1.01,fontsize=12); plt.tight_layout();
save_fig(fig, FOLDER, "01_spearman_rho_heatmap_all_tasks")

Saved: 01_spearman_rho_heatmap_all_tasks


In [ ]:
fig,axes=plt.subplots(1,4,figsize=(26,7))
for ai,task in enumerate(TASKS): hmap(nb7o[nb7o["Task"]==task],rater_names,rater_names,"Kendall_tau",axes[ai],f"{TASK_LABELS[task]}\nKendall \u03c4")
plt.suptitle("Kendall's \u03c4 — All Raters",y=1.01,fontsize=12); plt.tight_layout();
save_fig(fig, FOLDER, "02_kendall_tau_heatmap_all_tasks")

Saved: 02_kendall_tau_heatmap_all_tasks


In [ ]:
llm_r=["L1","L2","L3","L4"]; llm_colors={"L1":COLORS4[0],"L2":COLORS4[1],"L3":COLORS4[2],"L4":COLORS4[3]}
for human in ["H1","H2"]:
    fig,axes=plt.subplots(1,4,figsize=(20,6))
    for ai,task in enumerate(TASKS):
        ax=axes[ai]
        for llm in llm_r:
            pair_df=nb7c[((nb7c["Rater1"]==human)&(nb7c["Rater2"]==llm))|((nb7c["Rater1"]==llm)&(nb7c["Rater2"]==human))]
            pair_df=pair_df[pair_df["Task"]==task].copy(); pair_df["Cat"]=pair_df["Category"].map(CAT_SHORT)
            pair_df=pair_df.set_index("Cat").reindex(cats_short)
            ax.plot(range(len(cats_short)),pair_df["Spearman_rho"].tolist(),marker="o",label=RATER_LABELS[llm],color=llm_colors[llm],lw=1.5,ms=4)
        ax.set_xticks(range(len(cats_short))); ax.set_xticklabels(cats_short,fontsize=8)
        ax.set_ylabel("Spearman \u03c1"); ax.set_title(TASK_LABELS[task],fontsize=9); ax.axhline(0,color="gray",lw=0.8,ls="--"); ax.set_ylim(-1,1)
    axes[0].legend(title="LLM",fontsize=7); plt.suptitle(f"{RATER_LABELS[human]} vs LLM — Spearman \u03c1 by Category",y=1.01,fontsize=12); plt.tight_layout()
    save_fig(fig, FOLDER, f"03_spearman_{human}_vs_LLMs")

llm_names=[RATER_LABELS[r] for r in llm_r]
fig,axes=plt.subplots(1,4,figsize=(20,6))
for ai,task in enumerate(TASKS):
    ax=axes[ai]; td=nb7o[nb7o["Task"]==task]
    mat=pd.DataFrame(np.nan,index=llm_names,columns=llm_names)
    for _,row in td.iterrows():
        r1,r2=row["Rater1"],row["Rater2"]
        if r1 in llm_r and r2 in llm_r:
            mat.loc[RATER_LABELS[r1],RATER_LABELS[r2]]=row["Spearman_rho"]; mat.loc[RATER_LABELS[r2],RATER_LABELS[r1]]=row["Spearman_rho"]
    m2=mat.values.copy().astype(float); np.fill_diagonal(m2,1.0); mat=pd.DataFrame(m2,index=llm_names,columns=llm_names)
    sns.heatmap(mat.astype(float),annot=True,fmt=".2f",cmap="coolwarm",vmin=-1,vmax=1,linewidths=0.5,ax=ax,annot_kws={"size":9},square=True)
    ax.set_title(TASK_LABELS[task],fontsize=10); ax.set_xticklabels(ax.get_xticklabels(),rotation=45,ha="right",fontsize=8); ax.set_yticklabels(ax.get_yticklabels(),rotation=0,fontsize=8)
plt.suptitle("LLM vs LLM Spearman \u03c1",y=1.01,fontsize=12); plt.tight_layout();
save_fig(fig, FOLDER, "05_LLM_vs_LLM_spearman_heatmap")

Saved: 03_spearman_H1_vs_LLMs
Saved: 03_spearman_H2_vs_LLMs
Saved: 05_LLM_vs_LLM_spearman_heatmap


In [ ]:
for human in ["H1","H2"]:
    fig,axes=plt.subplots(1,4,figsize=(20,6))
    for ai,task in enumerate(TASKS):
        ax=axes[ai]
        for llm in llm_r:
            pair_df=nb7c[((nb7c["Rater1"]==human)&(nb7c["Rater2"]==llm))|((nb7c["Rater1"]==llm)&(nb7c["Rater2"]==human))]
            pair_df=pair_df[pair_df["Task"]==task].copy(); pair_df["Cat"]=pair_df["Category"].map(CAT_SHORT)
            pair_df=pair_df.set_index("Cat").reindex(cats_short)
            ax.plot(range(len(cats_short)),pair_df["Spearman_rho"].tolist(),marker="o",label=RATER_LABELS[llm],color=llm_colors[llm],lw=1.5,ms=4)
        ax.set_xticks(range(len(cats_short))); ax.set_xticklabels(cats_short,fontsize=8)
        ax.set_ylabel("Spearman \u03c1"); ax.set_title(TASK_LABELS[task],fontsize=9); ax.axhline(0,color="gray",lw=0.8,ls="--"); ax.set_ylim(-1,1)
    axes[0].legend(title="LLM",fontsize=7); plt.suptitle(f"{RATER_LABELS[human]} vs LLM — Spearman \u03c1 by Category",y=1.01,fontsize=12); plt.tight_layout()
    save_fig(fig, FOLDER, f"03_spearman_{RATER_LABELS[human]}_vs_LLMs_all_tasks")

Saved: 03_spearman_Human 1_vs_LLMs_all_tasks
Saved: 03_spearman_Human 2_vs_LLMs_all_tasks


In [ ]:
col_names=["#","Sentence_ID","Source_File","Source_Sentence","Gold_Category","LLM","Has_Errors","Error_Span","Annotated_Category","Description","Corrected_Sentence","H1_C1","H1_C2","H1_C3","H1_C4","H1_Total","H2_C1","H2_C2","H2_C3","H2_C4","H2_Total","L1_C1","L1_C2","L1_C3","L1_C4","L1_Total","L2_C1","L2_C2","L2_C3","L2_C4","L2_Total","L3_C1","L3_C2","L3_C3","L3_C4","L3_Total","L4_C1","L4_C2","L4_C3","L4_C4","L4_Total"]
mdf=pd.read_excel(MASTER,sheet_name="Master Eval Sheet",header=None,skiprows=2,names=col_names)
mdf=mdf[mdf["#"].notna()].copy(); mdf=mdf[mdf["#"].astype(str)!="TOTAL"].reset_index(drop=True)
for task in TASKS:
    fig,axes=plt.subplots(1,4,figsize=(18,5))
    for ai,llm in enumerate(llm_r):
        ax=axes[ai]; xv=mdf[f"H1_{task}"].tolist(); yv=mdf[f"{llm}_{task}"].tolist()
        ax.scatter(xv,yv,alpha=0.4,color=llm_colors[llm],s=20)
        ax.set_xlabel(f"Human 1 ({TASK_LABELS[task]})"); ax.set_ylabel(RATER_LABELS[llm]); ax.set_title(f"Human 1 vs {RATER_LABELS[llm]}")
        mx=1 if task in ["C1","C3"] else 2; ax.set_xlim(-0.2,mx+0.2); ax.set_ylim(-0.2,mx+0.2); ax.plot([-0.2,mx+0.2],[-0.2,mx+0.2],color="gray",lw=0.8,ls="--")
    plt.suptitle(f"Human 1 vs LLM Evaluators — {TASK_LABELS[task]}",y=1.02,fontsize=12); plt.tight_layout(); save_fig(fig, FOLDER, f"06_scatter_H1_vs_LLMs_{task}")

Saved: 06_scatter_H1_vs_LLMs_C1
Saved: 06_scatter_H1_vs_LLMs_C2
Saved: 06_scatter_H1_vs_LLMs_C3
Saved: 06_scatter_H1_vs_LLMs_C4
